# Accuracy summary — all districts (v2: `Observed Yield` / `Predicted Yield` columns)

Reads every district result workbook in `alldistricts/` and computes district-level MAE, RMSE, Pearson R, MAPE, a t-test and Willmott's index of agreement.

**Inputs:** `alldistricts/*.xlsx` (outputs of `yield_ml_*`)  
**Outputs:** `AllDistricts.xlsx`  
**Run after:** all `yield_ml_*` notebooks  

> Update the path variables in the first cells before running. See [`docs/`](../../docs/) for methodology and parameters.
>
> ⚠️ The index of agreement uses `|O|` where Willmott's formula has `|O − Ō|`. See `docs/known-issues.md`.

In [ ]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
import rasterio as rio

In [ ]:
from scipy import stats

In [ ]:
import geopandas as gpd

In [ ]:
import pandas as pd

In [ ]:
from shapely import Point
import numpy as np

In [ ]:
import os
from scipy.stats import pearsonr

In [ ]:
out_path=r'C:\Local\Desktop_previous\miscellaneous\Neha\YieldData\21Jan\alldistricts'

In [ ]:
def get_coord(shape):
    shapefile=pd.read_csv(shape)
    coords = [(x,y) for x, y in zip(shapefile.longitude, shapefile.latitude)]
    return coords

In [ ]:
def getRasterValue(image,coords):
    ras = rio.open(image)
    return [x[0] for x in ras.sample(coords)]

In [ ]:
def mape(y_true, y_pred): 
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100


In [ ]:
def indexofagreement(y_true, y_pred):
    if len(y_true)>1:
        muo = (y_true).mean()
        n=[]
        din=[]
        for d in range(len(y_true)):
            n.append((y_true[d]-y_pred[d])**2)
            din.append((abs(y_pred[d]-muo)+abs(y_true[d]))**2)
        ioa = sum(n)/sum(din)
    else:
        ioa=1
    return 1-ioa

In [ ]:
def accuracy_parameter(df):
    y_test=np.array(df['Observed Yield'])
    y_pred=np.array(df['Predicted Yield'])
    mae = mean_absolute_error(y_true=y_test,y_pred=y_pred)
    mse = mean_squared_error(y_true=y_test,y_pred=y_pred)
    rmse = mean_squared_error(y_true=y_test,y_pred=y_pred,squared=False)
    if len(y_test)>2:
        r2 = pearsonr(y_test,y_pred)[0]
    else:
        r2=0
    maper = mape(y_test,y_pred)
    test_mean=y_test.mean()
    pred_mean=y_pred.mean()
    t_stat, p_val = stats.ttest_ind( y_pred,y_test)
    ioa = indexofagreement(y_test,y_pred)
    
    return mae,mse,rmse,r2,maper,t_stat, p_val,ioa,test_mean, pred_mean

In [ ]:
dist=[]
param=[]
file = (r"C:\Users\PushkarGaur\Downloads\THUTHOOKUDI_MAIZE.xlsx")
df_i = pd.read_excel(file)
dist.append(file.split('.')[0].split('_')[0].split('\\')[-1])
param.append((accuracy_parameter(df_i)))
df=pd.DataFrame(param,columns=['MAE','MSE','RMSE','R','MAPE','T','p','IndexOfAgreement','Observed','Predicted'])
df['District']=dist
df=df.drop(columns=['MSE'])

In [ ]:
df

In [ ]:
df.to_excel(os.path.join(out_path,'THUTHOOKUDI.xlsx'))